# Isolation Forest

In [0]:
%run
Utils/configuration

imblearn is installed

Out[3]: DataFrame[]

In [0]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import SCORERS
from sklearn.svm import SVC
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.datasets import load_iris

## Carregar os dados e transformar

In [0]:
iris = load_iris()
df = pd.DataFrame(data=iris.data, 
                  columns=iris.feature_names)
df['species'] = iris.target
df['species'] = df['species'].map({0: 'Iris-virginica', 1:'Iris-setosa', 2:'Iris-versicolor'})
df = spark.createDataFrame(df)

In [0]:
df = df.withColumnRenamed('sepal length (cm)', "sepal_length")
df = df.withColumnRenamed('sepal width (cm)', "sepal_Width")
df = df.withColumnRenamed('petal length (cm)', "petal_length")
df = df.withColumnRenamed('petal width (cm)', "petal_width")

In [0]:
y = df.select("species").withColumn("code", when(df.species == "Iris-virginica", 0).when(df.species == "Iris-setosa", 1).otherwise(2))
y = np.array(y.select("code").collect())
X = df.select(["sepal_length", "sepal_Width", "petal_length", "petal_width"]).toPandas()

## Escolha de Parâmetros

In [0]:
param = {'n_estimators': [50, 100], 'max_samples': [ 1.0, 10, 100],
         'contamination': [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 1],
         'bootstrap': (True, False)}

In [0]:
def escol_param(X,y, estimator, param_grid, scoring='f1_micro', n_jobs=None, refit=True, cv=None, verbose=0, pre_dispatch='2*n_jobs', error_score=np.nan, return_train_score=False):
  gs = GridSearchCV(estimator= estimator , param_grid = param_grid, scoring=scoring, n_jobs=n_jobs, refit=refit, cv=cv, verbose=verbose, pre_dispatch=pre_dispatch, error_score=error_score, return_train_score=return_train_score)
  gs.fit(X, y)
  parametro = {}
  variavel = list(param.keys())
  for i in variavel:
    parametro[i] = gs.best_params_[i]
  return parametro

In [0]:
escol_param(X,y, estimator = IsolationForest(), param_grid = param)

Out[52]: {'n_estimators': 50,
 'max_samples': 1.0,
 'contamination': 0.01,
 'bootstrap': True}

# Methods
* modelos não supervisionados para detectar anomalias.

## ident_outliers_isolation_forest

* Isola as observações randomicamente selecionando as variáveis, e depois seleciona randomicamente um valor de corte entre o minimo e o maximo da variável selecionada.
* algoritmo para detecção de anomalias, ele retira tanto os outliers variados como também os multivariados.

### Documentação
---
https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/3749511783433280/command/3749511783433376

### Exemplo

In [0]:
modelo = ident_outliers_isolation_forest(df, 'species', n_estimators=100, max_samples= 1.0, contamination= 0.01, bootstrap=True)

In [0]:
display(df.filter("species = 'Iris-setosa'"))

sepal_length,sepal_Width,petal_length,petal_width,species
7.0,3.2,4.7,1.4,Iris-setosa
6.4,3.2,4.5,1.5,Iris-setosa
6.9,3.1,4.9,1.5,Iris-setosa
5.5,2.3,4.0,1.3,Iris-setosa
6.5,2.8,4.6,1.5,Iris-setosa
5.7,2.8,4.5,1.3,Iris-setosa
6.3,3.3,4.7,1.6,Iris-setosa
4.9,2.4,3.3,1.0,Iris-setosa
6.6,2.9,4.6,1.3,Iris-setosa
5.2,2.7,3.9,1.4,Iris-setosa


In [0]:
display(modelo.filter("species = 'Iris-setosa'"))

sepal_length,sepal_Width,petal_length,petal_width,species
6.4,3.2,4.5,1.5,Iris-setosa
6.9,3.1,4.9,1.5,Iris-setosa
5.5,2.3,4.0,1.3,Iris-setosa
6.5,2.8,4.6,1.5,Iris-setosa
5.7,2.8,4.5,1.3,Iris-setosa
6.3,3.3,4.7,1.6,Iris-setosa
4.9,2.4,3.3,1.0,Iris-setosa
6.6,2.9,4.6,1.3,Iris-setosa
5.2,2.7,3.9,1.4,Iris-setosa
5.9,3.0,4.2,1.5,Iris-setosa


In [0]:
display(df.filter("species = 'Iris-setosa'"))

sepal_length,sepal_Width,petal_length,petal_width,species
7.0,3.2,4.7,1.4,Iris-setosa
6.4,3.2,4.5,1.5,Iris-setosa
6.9,3.1,4.9,1.5,Iris-setosa
5.5,2.3,4.0,1.3,Iris-setosa
6.5,2.8,4.6,1.5,Iris-setosa
5.7,2.8,4.5,1.3,Iris-setosa
6.3,3.3,4.7,1.6,Iris-setosa
4.9,2.4,3.3,1.0,Iris-setosa
6.6,2.9,4.6,1.3,Iris-setosa
5.2,2.7,3.9,1.4,Iris-setosa


In [0]:
display(modelo.filter("species = 'Iris-setosa'"))

sepal_length,sepal_Width,petal_length,petal_width,species
6.4,3.2,4.5,1.5,Iris-setosa
6.9,3.1,4.9,1.5,Iris-setosa
5.5,2.3,4.0,1.3,Iris-setosa
6.5,2.8,4.6,1.5,Iris-setosa
5.7,2.8,4.5,1.3,Iris-setosa
6.3,3.3,4.7,1.6,Iris-setosa
4.9,2.4,3.3,1.0,Iris-setosa
6.6,2.9,4.6,1.3,Iris-setosa
5.2,2.7,3.9,1.4,Iris-setosa
5.9,3.0,4.2,1.5,Iris-setosa


In [0]:
display(df.filter("species = 'Iris-setosa'"))

sepal_length,sepal_Width,petal_length,petal_width,species
7.0,3.2,4.7,1.4,Iris-setosa
6.4,3.2,4.5,1.5,Iris-setosa
6.9,3.1,4.9,1.5,Iris-setosa
5.5,2.3,4.0,1.3,Iris-setosa
6.5,2.8,4.6,1.5,Iris-setosa
5.7,2.8,4.5,1.3,Iris-setosa
6.3,3.3,4.7,1.6,Iris-setosa
4.9,2.4,3.3,1.0,Iris-setosa
6.6,2.9,4.6,1.3,Iris-setosa
5.2,2.7,3.9,1.4,Iris-setosa


In [0]:
display(modelo.filter("species = 'Iris-setosa'"))

sepal_length,sepal_Width,petal_length,petal_width,species
6.4,3.2,4.5,1.5,Iris-setosa
6.9,3.1,4.9,1.5,Iris-setosa
5.5,2.3,4.0,1.3,Iris-setosa
6.5,2.8,4.6,1.5,Iris-setosa
5.7,2.8,4.5,1.3,Iris-setosa
6.3,3.3,4.7,1.6,Iris-setosa
4.9,2.4,3.3,1.0,Iris-setosa
6.6,2.9,4.6,1.3,Iris-setosa
5.2,2.7,3.9,1.4,Iris-setosa
5.9,3.0,4.2,1.5,Iris-setosa


In [0]:
display(df.filter("species = 'Iris-setosa'"))

sepal_length,sepal_Width,petal_length,petal_width,species
7.0,3.2,4.7,1.4,Iris-setosa
6.4,3.2,4.5,1.5,Iris-setosa
6.9,3.1,4.9,1.5,Iris-setosa
5.5,2.3,4.0,1.3,Iris-setosa
6.5,2.8,4.6,1.5,Iris-setosa
5.7,2.8,4.5,1.3,Iris-setosa
6.3,3.3,4.7,1.6,Iris-setosa
4.9,2.4,3.3,1.0,Iris-setosa
6.6,2.9,4.6,1.3,Iris-setosa
5.2,2.7,3.9,1.4,Iris-setosa


In [0]:
display(modelo.filter("species = 'Iris-setosa'"))

sepal_length,sepal_Width,petal_length,petal_width,species
6.4,3.2,4.5,1.5,Iris-setosa
6.9,3.1,4.9,1.5,Iris-setosa
5.5,2.3,4.0,1.3,Iris-setosa
6.5,2.8,4.6,1.5,Iris-setosa
5.7,2.8,4.5,1.3,Iris-setosa
6.3,3.3,4.7,1.6,Iris-setosa
4.9,2.4,3.3,1.0,Iris-setosa
6.6,2.9,4.6,1.3,Iris-setosa
5.2,2.7,3.9,1.4,Iris-setosa
5.9,3.0,4.2,1.5,Iris-setosa


In [0]:
modelo2 = ident_outliers_isolation_forest(df.select('sepal_Width','species').where('species = "Iris-setosa"'), 'species')

In [0]:
display(modelo2.filter("species = 'Iris-setosa'"))

sepal_Width,species
2.8,Iris-setosa
2.8,Iris-setosa
2.9,Iris-setosa
2.7,Iris-setosa
3.0,Iris-setosa
2.9,Iris-setosa
2.9,Iris-setosa
3.0,Iris-setosa
2.7,Iris-setosa
2.5,Iris-setosa
